### Re-ranking

In [1]:
import os
import sys
import joblib

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

model_name = "hgt"
dir = f"artifacts/{model_name}"
prefix = f"{model_name}_k_all"
eval_df = joblib.load(os.path.join(dir, f"{prefix}_eval_df.pkl"))
user_dps_df = joblib.load(os.path.join(dir, f"user_dps_df.pkl"))
feature_engineer = joblib.load(os.path.join(dir, f"feature_engineer.pkl"))

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


### MMR

In [2]:
from post_processing.mmr import MMR

mmr_reranker = MMR(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
)


Seed set to 42


In [3]:
mmr_result_df = mmr_reranker.rerank(
    top_k=20,
    theta=0.5,
)
mmr_result_df.head()

Preparing input DataFrame for MMR...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [00:30<00:00, 68.40it/s]


Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[2858, 1258, 7153, 1090, 293, 1, 778, 2571, 75...","[2858, 4226, 7153, 2571, 1, 260, 5952, 1258, 1...","[2058, 163, 2490, 45722, 1233, 110, 2959, 2571]"
1,78,"[1090, 7153, 2791, 8950, 6283, 1258, 4896, 285...","[1090, 2858, 4226, 288, 7153, 2791, 6870, 904,...","[5881, 37729, 44191, 4119, 6993, 8400, 50872]"
2,127,"[1090, 6283, 2791, 3328, 8950, 4896, 2406, 245...","[1090, 6283, 6870, 288, 3328, 2791, 2455, 5388...","[45726, 6958]"
3,170,"[7153, 1090, 1258, 2791, 293, 260, 6016, 1617,...","[7153, 260, 5952, 1090, 1258, 288, 1961, 750, ...","[4963, 1222, 3949, 4011, 2542, 8874, 44191, 45..."
4,175,"[2858, 1258, 7153, 1090, 2791, 8950, 36529, 1,...","[2858, 4226, 1090, 7153, 288, 2791, 1, 6870, 9...","[1921, 5995, 1913, 1419, 4927, 50068, 7700, 17..."


In [4]:
from common.eval import Evaluator
evaluator = Evaluator()

mmr_reranked_score_df = evaluator.evaluate(mmr_result_df, K=5)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=10)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=20)
mmr_reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.101383,0.011695,0.036047,0.130728,0.021286,0.034399,0.159580,0.036017,0.030547
std,20797.975208,0.255496,0.047872,0.090772,0.252282,0.059269,0.067780,0.240520,0.073539,0.049995
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.000000,0.000000,0.000000,0.289065,0.010127,0.100000,0.300405,0.047619,0.050000
max,71534.000000,1.000000,1.000000,0.600000,1.000000,1.000000,0.500000,1.000000,1.000000,0.400000


In [5]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=mmr_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 871.68it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.073409,0.857777,0.088780,0.833847,0.463454
std,595.969798,0.041265,0.071767,0.069832,0.082723,0.040465
min,0.000000,0.000000,0.504505,0.000000,0.357616,0.313183
25%,515.750000,0.042612,0.813890,0.034042,0.794689,0.437533
50%,1031.500000,0.071484,0.867115,0.082141,0.850912,0.465511
75%,1547.250000,0.100043,0.912133,0.133914,0.892665,0.491505
max,2063.000000,0.247990,0.992015,0.380398,0.976242,0.588261


In [6]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(mmr_result_df, k=10)
ils_df.describe()

,user,ILS@10
count,2064.000000,2064.000000
mean,35564.367733,0.094704
std,20797.975208,0.021354
min,75.000000,0.045946
25%,17798.500000,0.079681
50%,35054.000000,0.092348
75%,53331.000000,0.107772
max,71534.000000,0.164365


### DPA-RS

In [7]:
from post_processing.dpa_rs import DPA_RS

reranker = DPA_RS(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
    ground_truth_dps_df=user_dps_df
)


Seed set to 42


In [8]:
dpa_result_df = reranker.rerank(
    top_k=20,
    max_iter=100,
    random_state=42,
)
dpa_result_df.head()

Preparing input DataFrame for DPA-RS...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064
Combined DataFrame shape: (206400, 16)
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [10:44<00:00,  3.20it/s]

Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[5502, 231, 6283, 34405, 70, 1729, 6016, 2167,...","[2858, 4226, 7153, 2571, 1, 260, 5952, 1258, 1...","[2058, 163, 2490, 45722, 1233, 110, 2959, 2571]"
1,78,"[1213, 4878, 6283, 2571, 3285, 4720, 8368, 778...","[1090, 2858, 4226, 288, 7153, 2791, 6870, 904,...","[5881, 37729, 44191, 4119, 6993, 8400, 50872]"
2,127,"[4367, 2167, 4720, 3983, 51709, 2599, 543, 511...","[1090, 6283, 6870, 288, 3328, 2791, 2455, 5388...","[45726, 6958]"
3,170,"[2406, 3994, 590, 5110, 3114, 6283, 2599, 4979...","[7153, 260, 5952, 1090, 1258, 288, 1961, 750, ...","[4963, 1222, 3949, 4011, 2542, 8874, 44191, 45..."
4,175,"[2951, 2997, 1220, 47629, 36529, 4995, 6870, 1...","[2858, 4226, 1090, 7153, 288, 2791, 1, 6870, 9...","[1921, 5995, 1913, 1419, 4927, 50068, 7700, 17..."


In [9]:
from common.eval import Evaluator
evaluator = Evaluator()

reranked_score_df = evaluator.evaluate(dpa_result_df, K=5)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=10)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=20)
reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.098778,0.010665,0.035853,0.128722,0.017833,0.032074,0.157084,0.031102,0.028198
std,20797.975208,0.250774,0.037249,0.092120,0.253413,0.044664,0.064628,0.245480,0.061418,0.046595
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.299188,0.043478,0.050000
max,71534.000000,1.000000,0.500000,0.800000,1.000000,0.500000,0.600000,1.000000,0.600000,0.350000


In [10]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=dpa_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 865.18it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.169286,0.976440,0.429970,0.922400,0.624524
std,595.969798,0.056076,0.024266,0.133174,0.033967,0.036367
min,0.000000,0.000000,0.641608,0.000000,0.731173,0.484986
25%,515.750000,0.133897,0.970337,0.347330,0.905926,0.602839
50%,1031.500000,0.169822,0.983227,0.431820,0.928119,0.625389
75%,1547.250000,0.204177,0.990758,0.516080,0.946382,0.648738
max,2063.000000,0.446360,1.000000,0.858128,0.987240,0.762344


In [11]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(dpa_result_df, k=10)
ils_df.describe()

,user,ILS@10
count,2064.000000,2064.000000
mean,35564.367733,0.261233
std,20797.975208,0.068412
min,75.000000,0.094352
25%,17798.500000,0.210956
50%,35054.000000,0.252790
75%,53331.000000,0.307668
max,71534.000000,0.504775
